In [1]:
import pathlib

import plotly
import plotly.express as px
import plotly.graph_objects as go

import trapallada.analysis

So that `import`ed Python modules are automatically reloaded when modified

In [ ]:
%load_ext autoreload
%autoreload 2

So that *plotly* pictures can be exported to HTML

In [ ]:
plotly.offline.init_notebook_mode()

# Input data

Relative path and *Base* file name for the results (trained model). It should be a `str`.

In [4]:
input_files_stem = '../results/epochs/20000/4_sources/epochs'

It is turned into a `Pathlib`

In [ ]:
input_files_stem = pathlib.Path(input_files_stem)
input_files_stem

From the results directory,
- the `Dataset`,
- the headers of the corresponding files,
- the ML model,
- a `DataFrame` with results from training, 
- a `dict` with the parameters

In [ ]:
ds, positions_df, model, results_df, parameters, analysis_df = trapallada.analysis.load_results_dir(input_files_stem, analysis_also=True)

# Post-processing

Two new columns are added
- one combining the *tar* and *FIT* file, which univocally determines the signal,
- the other yielding the number of window *within* every (*tar* file, *FIT* file) combination.

In [7]:
analysis_df['# window'] = analysis_df.groupby(['tar file', 'source']).cumcount()
analysis_df['FIT file within tar'] = analysis_df['tar file'].str.replace('.tar', '') + '/' + analysis_df['source'].str.replace('.FTZ', '')
# analysis_df

Number of unique signals (before *windowing*)

In [ ]:
analysis_df['FIT file within tar'].nunique()

Number of windows per signal

In [ ]:
analysis_df.groupby('FIT file within tar').size()

For the sake of readability

In [ ]:
on_plot_hover_columns = ['index', 'source', 'cluster in projected space', 'main_type']
on_plot_hover_columns

# Results

A plot with **all** the windows for **all** the signals, colored by the latter.

In [ ]:
fig = px.scatter(analysis_df.reset_index(), x='tsne_1', y='tsne_2', color='FIT file within tar', hover_data=on_plot_hover_columns, opacity=0.5, width=1_200, height=800)
fig.show()

# Individual signals

In order to have the above axes in every picture

In [12]:
full_fig = fig.full_figure_for_development(warn=False)
x_range = full_fig.layout.xaxis.range
y_range = full_fig.layout.yaxis.range

A convenience function

In [13]:
def plot_fit_tar(name: str):
    df = analysis_df[analysis_df['FIT file within tar'] == name]

    # fig = px.scatter(df.reset_index(), x='tsne_1', y='tsne_2', text='# window', hover_data=on_plot_hover_columns, width=1_200, height=800)
    fig = px.scatter(df.reset_index(), x='tsne_1', y='tsne_2', color='# window', hover_data=on_plot_hover_columns, width=1_200, height=800)
    fig.update_layout(xaxis=dict(range=x_range), yaxis=dict(range=y_range))
    fig.update_traces(mode='lines+markers')

    # first point
    fig.add_trace(go.Scatter(
        x=[df['tsne_1'].iloc[0]],
        y=[df['tsne_2'].iloc[0]],
        mode='markers',
        marker=dict(size=12, color='black', symbol='square'),  # Mark the first point with a square
        showlegend=False
    ))

    # last
    fig.add_trace(go.Scatter(
        x=[df['tsne_1'].iloc[-1]],
        y=[df['tsne_2'].iloc[-1]],
        mode='markers',
        marker=dict(size=12, color='black', symbol='triangle-up'),
        showlegend=False
    ))
    
    fig.show()

    # return df

## Javi's paper

In [ ]:
df = plot_fit_tar('0605950101/P0605950101PNS003SRCTSR8001')

## Something else

In [ ]:
plot_fit_tar('0006010301/P0006010301PNS003SRCTSR8001')
# plot_fit_tar('0406541301/P0406541301PNS010SRCTSR8002')

## Yet another

In [ ]:
plot_fit_tar('0006010401/P0006010401PNS003SRCTSR8001')

## Yet another II

In [ ]:
plot_fit_tar('0840380201/P0840380201PNS003SRCTSR8004')

## Yet another III

In [ ]:
plot_fit_tar('0743630201/P0743630201PNS001SRCTSR8002')